In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

sp.init_printing(use_unicode=True)
z = sp.Symbol('z', complex=True)
z_inv = sp.Symbol('z^{-1}', complex=True)
n = sp.Symbol('n', integer=True)

print("=== LTI System Analysis (Problem 090202) ===")

H_z_inv = (1 - sp.Rational(1, 2) * z_inv**2) / ((1 - sp.Rational(1, 2) * z_inv) * (1 - sp.Rational(1, 4) * z_inv))
H_z_inv = sp.factor(H_z_inv)

print("\n1. System Transfer Function H(z^-1):")
display(H_z_inv)

H_pfe_z_inv = sp.apart(H_z_inv, z_inv)

print("\n2. Partial Fraction Expansion of H(z^-1):")
display(H_pfe_z_inv)

def inverse_z_from_pfe(expr, z_inv_symbol, n_symbol):
    terms = sp.Add.make_args(sp.expand(expr))
    h = 0
    for term in terms:
        num, den = sp.fraction(sp.cancel(term))
        if not den.has(z_inv_symbol):
            h += sp.simplify(num / den) * sp.KroneckerDelta(n_symbol, 0)
            continue
        quotient, remainder = sp.div(sp.Poly(num, z_inv_symbol), sp.Poly(den, z_inv_symbol))
        quotient, remainder = sp.simplify(quotient.as_expr()), sp.simplify(remainder.as_expr())
        if quotient != 0:
            quotient_poly = sp.Poly(quotient, z_inv_symbol)
            for (power,), coefficient in quotient_poly.terms():
                h += coefficient * sp.KroneckerDelta(n_symbol, power)
        if remainder == 0:
            continue
        proper_term = sp.cancel(remainder / den)
        roots = sp.solve(sp.fraction(proper_term)[1], z_inv_symbol)
        root = roots[0]
        a = sp.simplify(1 / root)
        A = sp.simplify(sp.limit(proper_term * (1 - a * z_inv_symbol), z_inv_symbol, root))
        h += A * a**n_symbol * sp.Heaviside(n_symbol)
    return sp.simplify(h)

h_n = inverse_z_from_pfe(H_pfe_z_inv, z_inv, n)

print("\n3. Analytical Closed-Form Impulse Response h[n]:")
display(h_n)


# ==============================================================================
# DIFFERENCE EQUATION — SYMBOLICALLY CONSTRUCTED FROM H(z^-1)
# ==============================================================================

num_H, den_H = sp.fraction(sp.cancel(H_z_inv))

den_H = sp.expand(den_H)
num_H = sp.expand(num_H)

y = sp.Function('y')
x = sp.Function('x')

den_poly = sp.Poly(den_H, z_inv)
num_poly = sp.Poly(num_H, z_inv)

lhs = sum(coef * y(n - power) for (power,), coef in den_poly.terms())
rhs = sum(coef * x(n - power) for (power,), coef in num_poly.terms())

leading = den_poly.coeff_monomial(1)

lhs = sp.simplify(lhs / leading)
rhs = sp.simplify(rhs / leading)

print("\n4. Difference Equation:")
display(sp.Eq(lhs, rhs))


def evaluate_h(expr, n_values):
    values = []
    for k in n_values:
        value = expr.subs(n, int(k))
        value = value.replace(sp.Heaviside, lambda x: sp.Integer(1) if x >= 0 else sp.Integer(0))
        value = value.replace(sp.KroneckerDelta, lambda x, y: sp.Integer(1) if x == y else sp.Integer(0))
        value = sp.simplify(value)
        values.append(float(sp.N(value)))
    return np.array(values)

explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>LTI System Analysis — Problem 090202</b><br>
• <b>Frequency Response:</b> Magnitude and phase response.<br>
• <b>Pole-Zero Map:</b> Poles at z = 0.5 and z = 0.25.<br>
• <b>Impulse Response:</b> Automatically obtained from the symbolic partial-fraction expansion.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

def plot_system_analysis():
    with out:
        out.clear_output(wait=True)
        fig = plt.figure(figsize=(12, 11))
        gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.7])
        
        ax_mag = fig.add_subplot(gs[0, 0])
        ax_pz = fig.add_subplot(gs[0, 1])
        ax_phase = fig.add_subplot(gs[1, 0])
        ax_h = fig.add_subplot(gs[2, :])

        omega = np.linspace(0, np.pi, 500)
        num_w = 1.0 - 0.5 * np.exp(-2j * omega)
        den_w = (1.0 - 0.5 * np.exp(-1j * omega)) * (1.0 - 0.25 * np.exp(-1j * omega))
        H_omega = num_w / den_w
        mag_db = 20 * np.log10(np.maximum(np.abs(H_omega), 1e-12))
        phase_deg = np.angle(H_omega, deg=True)

        ax_mag.plot(omega / np.pi, mag_db, 'b')
        ax_mag.set_ylabel('Magnitude (dB)', color='b')
        ax_mag.set_xlabel(r'Normalized Freq ($\pi$ rad/sample)')
        ax_mag.grid(True)
        ax_mag.set_title('Frequency Response (Magnitude)', fontsize=10, fontweight='bold')

        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-1.0, 1.0)
        ax_pz.set_ylim(-1.0, 1.0)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)
        theta = np.linspace(0, 2 * np.pi, 200)
        ax_pz.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label='Unit Circle')
        ax_pz.scatter([0.5, 0.25], [0, 0], s=140, color='red', marker='x', label='Poles')
        zero_val = np.sqrt(0.5)
        ax_pz.scatter([zero_val, -zero_val], [0, 0], s=120, facecolors='none', edgecolors='blue', linewidths=2, marker='o', label='Zeros')
        ax_pz.set_title('Pole-Zero Map of H(z)', fontsize=10, fontweight='bold')
        ax_pz.legend(loc='upper right')

        ax_phase.plot(omega / np.pi, phase_deg, 'r')
        ax_phase.set_ylabel('Phase (deg)', color='r')
        ax_phase.set_xlabel(r'Normalized Freq ($\pi$ rad/sample)')
        ax_phase.grid(True)
        ax_phase.set_title('Frequency Response (Phase)', fontsize=10, fontweight='bold')

        n_vec = np.arange(0, 15)
        h_vals = evaluate_h(h_n, n_vec)
        ax_h.stem(n_vec, h_vals, basefmt=" ")
        ax_h.set_title('Impulse Response h[n]', fontsize=10, fontweight='bold')
        ax_h.set_xlabel('n')
        ax_h.set_ylabel('h[n]')
        ax_h.grid(True)

        plt.tight_layout()
        plt.show()

plot_system_analysis()
display(out)